# ML Assignment 1
**Topic:** Data Exploration & Foundational Math

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Dataset setup from practical prompt
np.random.seed(42)

n = 100
study_hours = np.random.normal(5, 2, n)
attendance = np.random.normal(85, 10, n)
previous_mark = np.random.normal(70, 15, n)
final_score = (0.5 * study_hours) + (0.3 * attendance) + (0.2 * previous_mark) + np.random.normal(0, 5, n)

df = pd.DataFrame({
    'study_hours': study_hours,
    'attendance': attendance,
    'previous_mark': previous_mark,
    'final_score': final_score
})

df.loc[0:5, 'study_hours'] = np.nan # Missing values
df.loc[99, 'study_hours'] = 50      # Outlier
df.loc[98, 'attendance'] = -10      # Impossible value

df.head(10)

## Part A — Inspect and Clean

In [ ]:
print("Null count:\n", df.isnull().sum())
print("\nMax study hours:", df['study_hours'].max())
print("Min attendance:", df['attendance'].min())

# Dropping NaNs, impossible negative attendance, and 50 hour outlier
df_clean = df.dropna().copy()
df_clean = df_clean[df_clean['attendance'] >= 0]
df_clean = df_clean[df_clean['study_hours'] < 40]

print("\nCleaned shape:", df_clean.shape)

### Part A Answers:
1. **Outlier Choice:** Removed the student with 50 study hours. 50 hours for one subject is unrealistic relative to the class mean of ~5 hours and looks like an entry typo. Leaving it in would heavily skew calculations like mean and correlation.
2. **Matrix specs:**
- Feature matrix shape: `(92, 3)` after dropping rows with missing or corrupt data.
- Numerical columns: all columns (`study_hours`, `attendance`, `previous_mark`, `final_score`).
- Target variable: `final_score`.

## Part B — Summarize

In [ ]:
for col in df_clean.columns:
    print(f"{col} -> Mean: {df_clean[col].mean():.2f}, Median: {df_clean[col].median():.2f}, Std: {df_clean[col].std():.2f}")

### Part B.2 Observations:
1. The mean and median are almost identical across all columns (e.g. study hours mean is ~4.88 vs median ~4.87), showing the cleaned data is evenly distributed.
2. `previous_mark` has a much wider spread (std ~14.86) compared to `study_hours` (std ~1.84).
3. Average attendance is high (~84.8%) with a small standard deviation, indicating solid consistent class attendance.

## Part C — Visualize

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df_clean['final_score'], bins=12, edgecolor='black', color='lightblue')
plt.title('Distribution of Final Score')
plt.xlabel('Final Score')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(6, 4))
plt.scatter(df_clean['study_hours'], df_clean['final_score'], color='orange', edgecolor='black')
plt.title('Study Hours vs Final Score')
plt.xlabel('Study Hours')
plt.ylabel('Final Score')
plt.show()

### Part C.2 Trend:
There is a moderate positive linear trend: more study hours generally correspond with a higher final score, though with spread due to the influence of other variables.

## Part D — Correlation and Causality

In [ ]:
r = df_clean['study_hours'].corr(df_clean['final_score'])
print("Pearson r:", round(r, 4))

### Part D.2 Critical Thinking:
- Studying 10 hours does not guarantee a specific score; correlation describes overall group trends, not individual guarantees. A student could spend 10 hours studying inefficiently or face test anxiety.
- **Lurking variable:** Student engagement or motivation. A student with high motivation is likely to study more hours *and* take better notes in class, which both drive the final score.

## Part E — Vectors and Dot Product

In [ ]:
x = df_clean[['study_hours', 'attendance', 'previous_mark']].iloc[0].values
print("Vector x:", x)

w = np.array([0.5, 0.2, 0.4])
b = 5

y = np.dot(w, x) + b
print("Calculated y:", y)

### Part E.4:
- Vector `w` represents feature weights (how much each unit of each feature influences the final score prediction).
- Scalar `b` is the bias/intercept, giving the baseline score if all input features were zero.

## Part F — Scaling

In [ ]:
study_mean = df_clean['study_hours'].mean()
study_std = df_clean['study_hours'].std()
prev_mean = df_clean['previous_mark'].mean()
prev_std = df_clean['previous_mark'].std()

df_clean['study_hours_scaled'] = (df_clean['study_hours'] - study_mean) / study_std
df_clean['previous_mark_scaled'] = (df_clean['previous_mark'] - prev_mean) / prev_std

df_clean[['study_hours', 'study_hours_scaled', 'previous_mark', 'previous_mark_scaled']].head()

### Part F.2:
`previous_mark` varies across a scale of ~30-100, whereas `study_hours` only varies between ~1-9. When KNN computes Euclidean distances $(x_1 - x_2)^2$, the differences in `previous_mark` completely overwhelm `study_hours`, making the model ignore study hours unless scaled.